# 05 — OE4: Dinámica de shocks (VAR/VECM, IRF, FEVD)

Responde **PI4** (¿cómo y por cuánto responde el retorno a un shock del cobre/TC?) y aporta
evidencia para **H6** (dominancia global) vía descomposición de varianza.

- Si OE3 encontró cointegración -> **VECM**; si no -> **VAR** en retornos/diferencias.
- Identificación Cholesky por **orden de exogeneidad**: cobre -> externas -> locales -> acción.
- Robustez: IRF generalizadas (invariantes al orden).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config as C
from src import var_irf as vi

## 1. Construir el sistema en variables ESTACIONARIAS (retornos / diferencias)

In [ ]:
g = pd.read_parquet(C.DATA_INTERIM / 'raw_macro_global.parquet')
px = pd.read_parquet(C.DATA_INTERIM / 'raw_precios.parquet')
lf = pd.read_parquet(C.DATA_INTERIM / 'raw_macro_local_fred.parquet')

ret_cartera = np.log(px[['ANTO.L']].resample('ME').last()).diff()
d_cobre = np.log(g[['cobre']].resample('ME').last()).diff()
d_vix = g[['vix']].resample('ME').last().diff()
d_usdclp = np.log(lf[['usdclp']].resample('ME').last()).diff()

sistema = pd.concat([d_cobre, d_vix, d_usdclp, ret_cartera], axis=1).loc['2004':'2024'].dropna()
sistema.columns = ['d_cobre','d_vix','d_usdclp','retorno_cartera']
sistema.tail()

## 2. Selección de rezagos

In [ ]:
vi.seleccionar_rezagos(sistema, maxlags=12).round(3)

## 3. Estimar VAR con orden por exogeneidad

In [ ]:
orden = ['d_cobre','d_vix','d_usdclp','retorno_cartera']  # acción al final
res = vi.estimar_var(sistema, orden=orden, p=2)  # ajustar p segun seccion 2
print(res.summary())

In [ ]:
# Estabilidad del VAR (raíces dentro del círculo unitario)
print('VAR estable:', res.is_stable())

## 4. Funciones impulso-respuesta (IRF)
Respuesta del retorno de la cartera a un shock de 1 desv. en el cobre.

In [ ]:
irf = vi.irf(res, periodos=24, ortogonal=True)
fig = irf.plot(impulse='d_cobre', response='retorno_cartera', orth=True)
fig.suptitle('IRF: shock del cobre -> retorno de la cartera')
fig.savefig(C.FIGURES / 'oe4_irf_cobre_retorno.png', dpi=120)

In [ ]:
# Robustez: IRF generalizadas (no dependen del orden de Cholesky)
# statsmodels: usar orth=False para IRF no ortogonalizadas como contraste.
fig2 = irf.plot(impulse='d_usdclp', response='retorno_cartera', orth=True)
fig2.suptitle('IRF: shock del tipo de cambio -> retorno de la cartera')
fig2.savefig(C.FIGURES / 'oe4_irf_usdclp_retorno.png', dpi=120)

## 5. Descomposición de varianza (FEVD) — evidencia para H6

In [ ]:
tab_fevd = vi.tabla_fevd_variable(res, 'retorno_cartera', periodos=(1,6,12,24))
display(tab_fevd.round(3))
tab_fevd.to_csv(C.TABLES / 'oe4_fevd_retorno.csv')
# H6: sumar columnas 'globales' (d_cobre, d_vix) vs 'locales' (d_usdclp)
print('Fracción explicada por shocks GLOBALES vs LOCAL en h=12:')
print(' global :', round(tab_fevd.loc[12, ['d_cobre','d_vix']].sum(), 3))
print(' local  :', round(tab_fevd.loc[12, ['d_usdclp']].sum(), 3))

## 6. Causalidad de Granger

In [ ]:
for v in ['d_cobre','d_vix','d_usdclp']:
    stat, p = vi.granger_causalidad(res, v, 'retorno_cartera')
    print(f'{v:10s} -> retorno_cartera:  F={float(stat):8.3f}  p={float(p):.4f}')

## 7. Lectura (OE4)
- Forma y persistencia de la **IRF** al shock del cobre (¿positiva? ¿cuántos meses dura?) -> PI4.
- **FEVD**: ¿los shocks globales (cobre, VIX) explican más varianza del retorno que el local? -> H6.
- **Granger**: ¿qué variables anteceden al retorno?

Robustez obligatoria: reordenar Cholesky y/o usar IRF generalizadas; el resultado de H6 no
debe depender críticamente del orden.

Con esto quedan cubiertos OE1–OE5. Cierre integrado en el Capítulo 4 (discusión).